<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20261027/%E5%88%86%E9%A1%9E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 分類 ― ロジスティック回帰による2値分類

本資料では、教師あり学習の代表的な課題である**分類**を扱う。特に、2値分類の基本モデルである**ロジスティック回帰**について、考え方、モデル構築、予測確率、評価指標、係数の解釈までを一連の流れとして学ぶ。


#分類問題とは何か

機械学習の教師あり学習では、正解データを用いて入力と出力の関係を学習する。出力の種類によって、代表的には次のように分類される。

| 問題 | 予測する値 | 例 |
|---|---|---|
| 回帰 | 連続的な数値 | 売上、需要量、価格、所要時間 |
| 分類 | 離散的なカテゴリ | 解約する／しない、不正／正常、良性／悪性 |

分類のうち、カテゴリが2種類の問題を**2値分類**と呼ぶ。3種類以上のカテゴリを扱う問題を**多クラス分類**と呼ぶ。

##分類手法が活用される場面

分類は多くの業務で用いられる。代表例は以下のとおりである。

| 活用場面 | 目的変数の例 | 活用目的 |
|---|---|---|
| 顧客離反予測 | 解約する／しない | 離反可能性が高い顧客への施策を検討する |
| 与信判断 | 延滞する／しない | 融資や取引条件の判断を支援する |
| 不正検知 | 不正／正常 | 不審な取引を早期に検知する |
| 設備保全 | 故障する／しない | 予防保全の優先順位を決める |
| 医療データ分析 | 疾患あり／なし | 診断や検査の補助に用いる |
| スパム判定 | スパム／正常 | 不要なメッセージを自動分類する |

分類モデルの出力は意思決定を支援する情報であり、モデルの予測そのものが原因や因果関係を示すわけではない点に注意が必要である。

##代表的な分類手法

代表的な分類手法には次のようなものがある。

- ロジスティック回帰
- 決定木
- ランダムフォレスト
- k-近傍法
- サポートベクターマシン
- 勾配ブースティング
- ニューラルネットワーク

本講義では、構造が比較的単純で、予測確率と係数との関係を解釈しやすい**ロジスティック回帰**を扱う。ロジスティック回帰は名称に「回帰」を含むが、機械学習では分類モデルとして利用される。

#分析環境の準備

Google Colab 環境を想定する。日本語をグラフに表示するため `japanize-matplotlib` を使用する。

In [ ]:
%%capture
# Google Colab 等で初回のみ実行する
!pip -q install japanize-matplotlib

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import japanize_matplotlib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    RocCurveDisplay,
)

#乳がんデータの理解
この資料では、
scikit-learn に含まれる Breast Cancer Wisconsin (Diagnostic) データを用いる。このデータは569件の観測と30個の数値特徴量から構成され、目的変数は悪性（malignant）と良性（benign）の2クラスである。
なお、このデータでは **0=悪性、1=良性** となる点に注意が必要である。
分類問題として扱う上では、
**「どちらを1と定義するか」**
が評価指標の意味を左右するため、
最初に明確にすることが重要である。

In [ ]:
# データの読み込み
cancer = load_breast_cancer(as_frame=True)
# cancer.target に目的変数が格納
# cancer.data に説明変数が格納

# 説明変数
X = cancer.data.copy() # Xのdata型はDataframeとなる
# 目的変数
y = cancer.target.astype(int).copy() # yのdata型はSeriesである

# データフレーム
df = X.copy()
df['target'] = cancer.target.astype(int)

df.head()

##データ構造の確認

`info()` により、レコード数、データ型、欠損値の有無を確認する。機械学習では、モデル構築の前にデータの構造を把握することが必要である。

In [ ]:
df.info()

このデータでは30個の説明変数がすべて数値型であり、欠損値は含まれていない。説明変数は、細胞核の画像から計算された半径、質感、周囲長、面積などに関する特徴量である。

##目的変数の分布

目的変数のクラスごとの件数を確認する。
特定のクラスが極端に少ない場合には、
正解率だけでは性能を適切に評価できないことがある。

In [ ]:
class_count = y.value_counts().sort_index()
class_ratio = y.value_counts(normalize=True).sort_index()

pd.DataFrame({
    '件数': class_count,
    '割合': class_ratio
}).rename(index={0: '悪性(0)', 1: '良性(1)'})

In [ ]:
target_labels = y.map({0: '悪性(0)', 1: '良性(1)'})
target_counts = target_labels.value_counts().reindex(['悪性(0)', '良性(1)'])

plt.figure(figsize=(6, 3))
plt.bar(target_counts.index, target_counts.values)
plt.xlabel('診断クラス')
plt.ylabel('件数')
plt.title('目的変数の分布')
plt.show()


##説明変数と目的変数の関係

単一の説明変数だけでもクラス間の傾向が説明できるか確認する。
一般には、分布が重なる領域があり一変数だけで完全に分類することは難しいため、
複数の説明変数を組み合わせて予測を行う。

In [ ]:
plot_df = df.copy()
plot_df['診断'] = plot_df['target'].map({0: '悪性', 1: '良性'})

plt.figure(figsize=(7, 4))
sns.histplot(data=plot_df, x='mean radius', hue='診断', kde=True, element='step')
plt.title('mean radius の分布')
plt.show()

#学習用データとテスト用データへの分割

予測モデルは、学習に使用したデータだけで評価すると過大評価されやすい。そのため、モデル構築に用いる**学習用データ**と、最終的な性能確認に用いる**テスト用データ**を分離する。

本講義ではデータの80%を学習用、20%をテスト用とする。
比率としては8:2や7:3がよく使われる設定である。

さらに、分類ではクラス比率を保つため
**層化分割** を行う。これには、`stratify=y` を指定する。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('全体:', X.shape)
print('学習用:', X_train.shape)
print('テスト用:', X_test.shape)

In [ ]:
pd.DataFrame({
    '全体': y.value_counts(normalize=True).sort_index(),
    '学習用': y_train.value_counts(normalize=True).sort_index(),
    'テスト用': y_test.value_counts(normalize=True).sort_index(),
}).rename(index={0: '悪性(0)', 1: '良性(1)'})

##データリーケージに注意する

**データリーケージ**とは、本来モデル学習時には利用できない情報が学習処理に入り込み、性能が実際より高く見える問題である。

典型例は、データの標準化を行う際に、全データを使って平均値や標準偏差を計算した後に、学習用とテスト用へ分割する処理である。
テスト用データの情報が前処理を通じて学習側へ漏れるためである。

したがって、標準化や欠損値補完などの「データからパラメータを学ぶ前処理」では、
原則として**学習用データだけで `fit`** し、その結果をテスト用データへ適用する。

#説明変数の標準化

標準化とは、各説明変数を次式で変換する処理である。

$$
 x' = \frac{x-\mu}{\sigma}
$$

ここで、$\mu$ は学習用データにおける平均、$\sigma$ は学習用データにおける標準偏差である。

ロジスティック回帰に標準化が数学的に必須というわけではない。
しかし、scikit-learn の `LogisticRegression` は既定で正則化を利用するため、説明変数の尺度をそろえることには大きな意味がある。
また、数値計算の安定性や収束性能の向上や係数の比較可能性の点でも、標準化が有用である。

In [ ]:
scaler = StandardScaler()

# 学習用データだけで平均・標準偏差を学習する
X_train_scaled = scaler.fit_transform(X_train)

# テスト用データに変換を適用する
X_test_scaled = scaler.transform(X_test)

In [ ]:
scaled_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
scaled_train_df.describe().loc[['mean', 'std']].T.head(10)

`StandardScaler` は各変数を平均0、標準偏差1となるように変換したことが確認できる。

#ロジスティック回帰の仕組みと学習

##変数の線形結合によるスコア値とロジスティック関数

ロジスティック回帰では、まず説明変数の線形結合を計算する。

$$
 z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p
$$
ここで、$\beta_0$ は切片、$\beta_1,\ldots,\beta_p$ は回帰係数である。

次に、$z$ をロジスティック関数（シグモイド関数）へ入力することで、
0より大きく1未満となる数値に変換を行う。

$$
 p = \sigma(z) = \frac{1}{1+e^{-z}}
$$

この $p$ を、本講義では「ラベルが 1 （すなわち**良性**）である確率 $P(y=1|X)$」として解釈する。

In [ ]:
def logistic_function(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-7, 7, 200)
p = logistic_function(z)

plt.figure(figsize=(7, 4))
plt.plot(z, p)
plt.axhline(0.5, linestyle='--')
plt.axvline(0, linestyle='--')
plt.xlabel('z')
plt.ylabel('P(y=1 | X)')
plt.title('ロジスティック関数')
plt.grid(True)
plt.show()

上の図から、
$z$ が大きくなるほど $y=1$ の確率は1に近づき、$z$ が小さくなるほど0に近づくことが分かる。また、$z=0$ のとき予測確率は0.5である。

ロジスティック回帰では、観測された正解ラベルをよく説明するように係数を推定する。
統計学的には最尤法と結び付けて説明でき、
以下の対数損失（log loss）関数を最小化する問題として理解できる。
$$
\text{LogLoss} = - \frac{1}{n} \sum_{i=1}^{n} \{ y_i \log p_i + (1-y_i) \log( 1-p_i) \}
$$
ただし、$y_i$ はデータ $i$ の正解ラベル、$p_i$ は予測された確率である。

また、
scikit-learn の `LogisticRegression` では、
デフォルトで **L2正則化** が適用される。
正則化は係数が極端に大きくなることを抑え過学習の抑制や数値安定性に役立つ。

##モデルの学習

`fit()` を使って学習する。
> モジュール仕様として、目的変数 `y_train` は1次元の `Series` として渡す必要がある。

`max_iter=1000` と設定することで、内部での計算回数上限を増やし、
損失関数の最小化ができるだけ正常に終了するように設定する。

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

##係数の確認

標準化後の説明変数を用いているため、係数の符号は次のように解釈できる。

- 係数が正：その変数が大きいほど、他の条件が同じなら良性（1）側の対数オッズが大きくなる。
- 係数が負：その変数が大きいほど、他の条件が同じなら悪性（0）側の対数オッズが大きくなる。

ただし、係数は**因果効果を表すものではない**。また、このデータには相関の強い特徴量が複数あるため、係数の大きさだけを「重要度」とみなすのも慎重であるべきである。

### 対数オッズ（Log-Odds）とは

ロジスティック回帰の仕組みを深く理解するために
、**オッズ（Odds）**と**対数オッズ（Log-Odds）**
について説明を加える。

#### 1. オッズ（Odds）
オッズとは、「ある事象が起こる確率 $p$」と「起こらない確率 $1-p$」の比率である。
$$
\text{オッズ} = \frac{p}{1-p}
$$
例えば、良性（1）である確率 $p$ が $0.8$（80%）の場合、良性にならない確率 $1-p$ は $0.2$（20%）となり、オッズは $\frac{0.8}{0.2} = 4$ となります。これは「良性になる可能性が、悪性になる可能性の4倍高い」ことを意味する。

#### 2. 対数オッズ（Log-Odds / ロジット）
オッズの自然対数をとったものを
**対数オッズ（ロジット: logit）**
と呼ぶ。
$$
\text{対数オッズ} = \ln\left(\frac{p}{1-p}\right)
$$

ロジスティック回帰では、この対数オッズを説明変数の線形結合で
表現していると解釈できる。

$$
\ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p
$$

したがって、標準化された説明変数 $x_i$ が $1$ 単位増加したとき、他の条件が同じであれば、対数オッズが $\beta_i$ だけ変化することになる（オッズ比で表すと $e^{\beta_i}$ 倍になる）。

In [ ]:
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient')

coef_df.head(), coef_df.tail()

In [ ]:
plot_coef = coef_df.sort_values('coefficient')

plt.figure(figsize=(8, 10))
plt.barh(plot_coef['feature'], plot_coef['coefficient'])
plt.axvline(0, linewidth=1)
plt.title('標準化後のロジスティック回帰係数')
plt.show()


##予測確率とクラス予測

`predict_proba()` は各クラスの予測確率を返す。`predict()` は予測確率をクラスへ変換した結果を返す。

2値分類では、既定では陽性クラス（1としたクラスのこと）の確率が0.5以上なら1、
0.5未満なら0と判定する。
ここで重要なのは、**モデルはまず確率を計算し、その後に閾値でクラスへ変換している**ことである。

In [ ]:
# 良性クラス（1）の予測確率
p_test = model.predict_proba(X_test_scaled)[:, 1]

# 既定の閾値0.5によるクラス予測
y_pred = model.predict(X_test_scaled)

pd.DataFrame({
    '実際': y_test.to_numpy(),
    '良性確率': p_test,
    '予測': y_pred
}).head(10)

#分類モデルの評価

分類モデルの評価指標は、正解率だけではない。

まず、以下のように、予測結果を整理した表を
**混同行列（confusion matrix）** と呼ぶ。

|  | 予測：陰性(ラベル0) | 予測：陽性(ラベル1) |
|---|---:|---:|
| 実際：陰性(ラベル0) | TN | FP |
| 実際：陽性(ラベル1) | FN | TP |

- **TP (True Positive)**：陽性を陽性と予測した件数
- **TN (True Negative)**：陰性を陰性と予測した件数
- **FP (False Positive)**：陰性を陽性と予測した（間違った陽性）件数
- **FN (False Negative)**：陽性を陰性と予測した（間違った陰性）件数

ただし、特に医学の分野では、
**Positive** という呼び方は検査結果などが陽性（すなわち悪性）を意味し
ラベルとして１を付与、
**Negative** とは陰性の場合で（すなわち良性）を意味し、
ラベル０を付与することが多い。

扱っているデータの場合では、０が悪性、１が良性となる点に注意が必要である。


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['悪性(0)', '良性(1)']
).plot(cmap=None)
plt.title('混同行列')
plt.show()

##代表的な評価指標

### 正解率 Accuracy

$$
Accuracy = \frac{TP+TN}{TP+TN+FP+FN}
$$

全体のうち正しく分類できた割合である。
この指標は直感的であるが、
クラス不均衡が大きい場合には高い値でも実用上十分とは限らない。
例えば、悪性となるクラスが全体の1%であれば、
無条件に良性と予測する方法でも正解率は99%となる。

### 適合率 Precision

$$
Precision = \frac{TP}{TP+FP}
$$

陽性と予測したもののうち、実際に陽性であった割合である。

### 再現率 Recall

$$
Recall = \frac{TP}{TP+FN}
$$

実際の陽性をどの程度見逃さずに検出できたかを表す。
本講義の設定では「実際に良性であるデータのうち、良性と検出できた割合」
である。

### F1

$$
F1 = \frac{2 \times Precision \times Recall}{Precision+Recall}
$$

適合率と再現率の調和平均である。

### ROC-AUC
モデルの順位付け能力を評価する指標である。
実際の陽性と陰性のデータを一つずつ選んだ際に、
陽性のデータが陰性のデータより高い予測値（確率）となる割合とみなすことができる指標である。
したがって、1に近いほど高性能で、0.5付近はランダムな順位付けに近い。

In [ ]:
metrics = pd.Series({
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, p_test),
})

metrics

In [ ]:
RocCurveDisplay.from_predictions(y_test, p_test)
plt.title('ROC曲線')
plt.show()

##正解率だけでは不十分である理由

例えば陽性が全体の1%とする。
このデータでは、すべてを陰性と予測するだけで正解率が
99%になる。しかし、このモデルは陽性を1件も検出できない。

したがって、評価指標は目的に応じて選ぶ必要がある。

- 見逃しを減らしたい場合：再現率を重視する。
- 誤警報を減らしたい場合：適合率を重視する。
- 両者のバランスを見たい場合：F1を利用する。
- 閾値に依存しない順位付け性能を見たい場合：ROC-AUCなどを利用する。

実務では、誤分類による損失や業務上の制約を踏まえて指標を決める必要がある。

##判定閾値を変更する

既定の0.5は絶対的な正解ではない。目的に応じて閾値を変更できる。一般に閾値を下げると陽性と判定する件数が増え、再現率は上がりやすい一方、適合率は下がりやすい。

In [ ]:
def evaluate_threshold(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    return {
        'threshold': threshold,
        'accuracy': accuracy_score(y_true, pred),
        'precision': precision_score(y_true, pred, zero_division=0),
        'recall': recall_score(y_true, pred, zero_division=0),
        'f1': f1_score(y_true, pred, zero_division=0),
    }

threshold_result = pd.DataFrame([
    evaluate_threshold(y_test, p_test, t)
    for t in [0.3, 0.5, 0.7]
])

threshold_result

閾値は評価用データを見ながら都合よく調整すると過学習につながる。実務では、学習用データ内の交差検証や別の検証用データで閾値を選び、最終テスト用データは最後の評価に残すことが望ましい。

#正則化 ― 過学習を抑える

ロジスティック回帰では、学習データに対して対数損失（Log Loss）が小さくなるように係数を学習する。しかし、学習データへの適合だけを強く求めると、特定のデータに過度に適合し、未知データに対する性能が低下する場合がある。この現象を
**過学習（overfitting）**
という。

正則化（regularization）は、予測誤差だけでなく**係数の大きさにもペナルティを与える**ことで、モデルが必要以上に複雑になることを抑える方法である。

考え方を概念的に表すと、次のようになる。

$$
\text{目的関数}
=
\text{予測誤差}
+
\lambda \times \text{モデルの複雑さに対するペナルティ}
$$

正則化が強すぎるとモデルが単純になりすぎ、学習データにも十分に適合できない
**過少適合（underfitting）**
が生じる。一方、正則化が弱すぎると過学習しやすくなる。

したがって正則化では、**学習データへの適合度とモデルの単純さのバランス**を取ることが重要である。


##L2正則化

L2正則化では、係数の2乗和をペナルティとして加える。

$$
\text{LogLoss}
+
\lambda \sum_{j=1}^{p}\beta_j^2
$$

L2正則化には次の特徴がある。

- 大きな係数ほど強いペナルティを受ける。
- 多くの係数を全体的に0へ近づける方向に働く。
- 相関した説明変数が存在する場合にも、複数の変数を残しながら係数を抑える傾向がある。
- scikit-learn の `LogisticRegression` ではL2正則化が標準的な設定である。

L2正則化は、不要な複雑さを抑えながら多数の特徴量を利用したい場合に適している。


##L1正則化

L1正則化では、係数の絶対値の和をペナルティとして加える。

$$
\text{LogLoss}
+
\lambda \sum_{j=1}^{p}|\beta_j|
$$

L1正則化には次の特徴がある。

- 一部の係数を**完全に0**にすることがある。
- 係数が0になった特徴量は予測に寄与しなくなる。そのため、特徴量選択に近い効果が得られる。
- 説明変数が非常に多く、その一部だけが有用と考えられる場合に利用されることがある。

ただし、L1正則化によって選択された変数が必ず「本質的に重要な変数」であるとは限らない。相関の強い説明変数が複数ある場合には、どの変数が残るかがデータや設定によって変化することがある。


##L1とL2の違い

| 観点 | L1正則化 | L2正則化 |
|---|---|---|
| ペナルティ | $\sum |\beta_j|$ | $\sum \beta_j^2$ |
| 係数への作用 | 一部を0にしやすい | 全体的に小さくする |
| 特徴量選択 | 近い効果がある | 基本的に変数を残す |
| 解の特徴 | 疎（sparse）になりやすい | 滑らかに縮小されやすい |
| 主な用途 | 変数数を減らしたい場合 | 過学習を抑え安定化したい場合 |

どちらが常に優れているわけではない。モデルの目的、データ量、説明変数の数や相関構造などによって適切な方法は異なる。


##scikit-learn の `C` と正則化強度

scikit-learn の `LogisticRegression` では、`C` は**正則化強度の逆方向を表すパラメータ**である。

- `C` が小さい → 正則化が強い
- `C` が大きい → 正則化が弱い


| `C` | 正則化の強さ | モデルの傾向 |
|---:|---|---|
| 0.001 | 非常に強い | 係数が強く縮小される |
| 0.01 | 強い | 比較的単純なモデルになる |
| 0.1 | やや強い | 係数が抑えられる |
| 1 | 基準となる代表的な値 | 中程度 |
| 10 | 弱い | 学習データへの適合が強くなる |
| 100 | 非常に弱い | 過学習の可能性が高まる場合がある |

最適な `C` はデータによって異なるため、
交差検証によって候補を比較する必要がある。


##正則化と標準化の関係

正則化は**係数の大きさ**にペナルティを与える。そのため、説明変数の尺度が大きく異なると、係数の大きさを公平に扱えなくなる。

例えば、

- 年齢：20～80
- 年収：200～2,000
- 検査値：0.001～0.1

のように尺度が異なると、同じ程度の予測効果を持つ変数であっても必要となる係数の大きさが異なるため、ペナルティーを正しく与えることができない。
そこで、正則化を利用する線形モデルでは、一般に標準化などによって説明変数の尺度をそろえることが重要である。

本講義でロジスティック回帰の前に変数の標準化を行った理由の一つが、この正則化との関係である。


## 正則化項`C` を変えると係数と性能はどう変わるか

乳がんデータを用いて、L2正則化の強さを変化させる。ここではすでに学習用データだけから標準化した `X_train_scaled` と `X_test_scaled` を使用する。

各 `C` について次の値を確認する。

- 学習用データのAccuracy
- テスト用データのAccuracy
- 係数ベクトルの大きさ（L2ノルム）

正則化を強くすると係数は小さくなる。一方、予測性能が必ず向上するわけではない点にも注目する。


In [ ]:
C_values = np.logspace(-3, 2, 6)

regularization_results = []

for C in C_values:
    l2_model = LogisticRegression(
        C=C,
        max_iter=5000
    )
    l2_model.fit(X_train_scaled, y_train)

    regularization_results.append({
        'C': C,
        'Train Accuracy': l2_model.score(X_train_scaled, y_train),
        'Test Accuracy': l2_model.score(X_test_scaled, y_test),
        '係数L2ノルム': np.linalg.norm(l2_model.coef_[0])
    })

regularization_df = pd.DataFrame(regularization_results)
regularization_df


In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogx(
    regularization_df['C'],
    regularization_df['Train Accuracy'],
    marker='o',
    label='Train Accuracy'
)
plt.semilogx(
    regularization_df['C'],
    regularization_df['Test Accuracy'],
    marker='o',
    label='Test Accuracy'
)
plt.xlabel('C')
plt.ylabel('Accuracy')
plt.title('正則化強度とAccuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogx(
    regularization_df['C'],
    regularization_df['係数L2ノルム'],
    marker='o'
)
plt.xlabel('C')
plt.ylabel('係数のL2ノルム')
plt.title('正則化強度と係数の大きさ')
plt.grid(True)
plt.show()


`C` が小さいほど係数の大きさが抑えられることを確認する。

ただし、テストデータのAccuracyが `C` とともに単調に改善するとは限らない。正則化が強すぎれば過少適合となり、弱すぎれば過学習が起こる可能性がある。
適切な強さを選ぶ必要がある。


##L1正則化による係数の疎化を確認する

次に、
L1正則化による係数の疎化を確認する。
scikit-learnでは、バージョンによって正則化の指定方法が変更されている。本教材では、Colab上の複数のscikit-learnバージョンで動作しやすいように、バージョンを確認してL1/L2モデルを生成する補助関数を用意する。

- scikit-learn 1.8以降：`l1_ratio=1.0` をL1、`l1_ratio=0.0` をL2として指定する。
- それより前のバージョン：`penalty='l1'` / `penalty='l2'` を利用する。

また、solver には、L1とL2の両方を扱える `liblinear` を指定する。


In [ ]:
import sklearn
from packaging.version import Version

In [ ]:
def make_regularized_logistic(C=1.0, regularization='l2', max_iter=5000):
    # scikit-learn のバージョン差を吸収してL1/L2モデルを生成する
    common_params = {
        'C': C,
        'solver': 'liblinear',
        'max_iter': max_iter,
        'random_state': 42
    }

    if Version(sklearn.__version__) >= Version('1.8'):
        l1_ratio = 1.0 if regularization == 'l1' else 0.0
        return LogisticRegression(
            l1_ratio=l1_ratio,
            **common_params
        )
    else:
        return LogisticRegression(
            penalty=regularization,
            **common_params
        )

print('scikit-learn version:', sklearn.__version__)


In [ ]:
l1_model = make_regularized_logistic(
    C=0.1,
    regularization='l1'
)
l2_model = make_regularized_logistic(
    C=0.1,
    regularization='l2'
)

l1_model.fit(X_train_scaled, y_train)
l2_model.fit(X_train_scaled, y_train)

coef_compare = pd.DataFrame({
    'feature': X.columns,
    'L1係数': l1_model.coef_[0],
    'L2係数': l2_model.coef_[0]
})

zero_threshold = 1e-10

print(
    'L1でほぼ0となった係数数:',
    (np.abs(coef_compare['L1係数']) < zero_threshold).sum()
)
print(
    'L2でほぼ0となった係数数:',
    (np.abs(coef_compare['L2係数']) < zero_threshold).sum()
)

coef_compare.head(10)


In [ ]:
coef_compare['L1絶対値'] = coef_compare['L1係数'].abs()
top_l1 = coef_compare.sort_values('L1絶対値', ascending=False).head(15)

plt.figure(figsize=(8, 6))
plt.barh(top_l1['feature'], top_l1['L1係数'])
plt.axvline(0, linewidth=1)
plt.title('L1正則化で残った主な係数')
plt.xlabel('係数')
plt.gca().invert_yaxis()
plt.show()


L1正則化では一部の係数が0になりやすいことを確認できる。この性質を
**sparsity**という。

ただし、係数が0になったからといって、その変数が業務上・科学上まったく重要ではないとは限らない。正則化による選択結果は、説明変数間の相関、標本数、正則化強度などの影響を受ける。


#交差検証・Pipeline・ハイパーパラメータ探索

##最適な `C` をどのように決めるか

テスト用データを見ながら `C` を選ぶと、テストデータに対しても過学習してしまう。そのため、ハイパーパラメータである `C` は**学習用データの内部で交差検証を行って選択する**。

本講義では次の流れを採用する。

1. 最初に学習用データと最終テスト用データを分離する。
1. 学習用データ内部で5-fold交差検証によって複数の `C` を比較する。
1. 交差検証で最も良かった設定 `C` を選択する。
1. 最終テスト用データで評価して性能とする。

この手順により、モデル選択と最終評価を分離できる。


##Pipeline 処理の利用

交差検証では、各foldの検証データの情報が学習処理へ漏れないようにする必要がある。

標準化を交差検証の外側で実施すると、全学習用データの平均・標準偏差が各foldの前処理に使われ、検証foldの情報がわずかに漏れる。

そこで、`StandardScaler` と `LogisticRegression` を `Pipeline` にまとめ、Pipeline全体を `GridSearchCV` に渡す。
これにより、各foldごとに

1. fold内の学習データだけで標準化を学習する。
2. fold内の学習データでロジスティック回帰を学習する。
3. fold内の検証データで性能を評価する。

という処理が簡単に実現できる。


In [ ]:
pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10, 100]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print('best C =', grid.best_params_['logisticregression__C'])
print('best CV ROC-AUC =', grid.best_score_)

In [ ]:
# 最良モデルを、最後まで触っていなかったテスト用データで評価する
best_model = grid.best_estimator_
best_prob = best_model.predict_proba(X_test)[:, 1]
best_pred = best_model.predict(X_test)

pd.Series({
    'Accuracy': accuracy_score(y_test, best_pred),
    'Precision': precision_score(y_test, best_pred),
    'Recall': recall_score(y_test, best_pred),
    'F1': f1_score(y_test, best_pred),
    'ROC-AUC': roc_auc_score(y_test, best_prob),
})

##【再確認】Pipelineを利用する意義

`Pipeline` は「前処理 → モデル学習」を一つの処理としてまとめる仕組みである。主な利点は以下のとおりである。

1. 学習用データだけで前処理を `fit` するため、リーケージを防ぎやすい。
2. 予測時にも学習時と同じ前処理を確実に適用できる。
3. 交差検証やハイパーパラメータ探索と組み合わせやすい。
4. 実運用時の処理を再現しやすい。

初学者向けには前処理を個別に実行して仕組みを理解することも重要であるが、実務コードではPipelineの利用を推奨する。

#まとめ

本講義の重要事項は以下のとおりである。

1. 分類はカテゴリを予測する教師あり学習であり、2値分類と多クラス分類がある。
2. ロジスティック回帰は予測確率をロジスティック関数で表現する分類モデルである。
3. モデル構築前に、目的変数のクラス定義と分布を確認することが重要である。
4. 学習用データとテスト用データは分離し、分類では必要に応じて層化分割を行う。
5. 標準化はロジスティック回帰に数学的に必須ではないが、正則化や数値安定性の観点から有用である。
6. 前処理は学習用データだけで `fit` し、テスト用データの情報を漏らしてはならない。
7. `predict_proba()` で確率を求め、閾値によって最終的なクラスを決める。
8. 正解率だけでなく、適合率、再現率、F1、ROC-AUC、混同行列を目的に応じて使い分ける。
9. 閾値を変更すると適合率と再現率のトレードオフが変化する。
10. 係数や予測結果は因果関係を直接示すものではない。
11. 実務ではPipelineを使い、前処理とモデルを一体化することが有効である。

# 参考資料

- scikit-learn: Logistic Regression  
  https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
- scikit-learn: `LogisticRegression`  
  https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- scikit-learn: `StandardScaler`  
  https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
- scikit-learn: `train_test_split`  
  https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- scikit-learn: Classification metrics  
  https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics
- scikit-learn: Breast Cancer Wisconsin (Diagnostic) dataset  
  https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html
- scikit-learn: Pipelines and composite estimators  
  https://scikit-learn.org/stable/modules/compose.html